# Notebook para ejercicios 4, 5 y 6

In [88]:
include("/Users/rafa/Documents/Uni/Cuatri4/machine_learning/FAA_practices/src/boletin_1/45159263M_49918198X_48118738R_54153358L.jl")

trainClassANN (generic function with 2 methods)

## Ejercicio 4

In [5]:
function confusionMatrix(outputs::AbstractArray{Bool,1}, targets::AbstractArray{Bool,1})

    true_positives = sum(outputs .& targets)
    false_positives = sum(outputs .& .!targets)
    true_negatives = sum(.!outputs .& .!targets)
    false_negatives = sum(.!outputs .& targets)

    accuracy = (true_positives + true_negatives) / length(outputs) # precision
    fail_rate = (false_positives + false_negatives) / length(outputs) # tasa de fallo
    recall = (true_positives == 0 && false_negatives == 0) ? 1 : true_positives / (true_positives + false_negatives) # sensibilidad
    especificity = (true_negatives == 0 && false_positives == 0) ? 1 : true_negatives / (true_negatives + false_positives) # especificidad
    precision = (true_positives == 0 && false_positives == 0) ? 1 : true_positives / (true_positives + false_positives) # valor predictivo positivo
    npv = (true_negatives == 0 && false_negatives == 0) ? 1 : true_negatives / (true_negatives + false_negatives) # valor predictivo negativo
    f1 = (recall == 0 && precision == 0) ? 0 : 2 * (precision * recall) / (precision + recall) # f1 score
    confussion_matrix = [true_negatives false_positives; false_negatives true_positives] # matriz de confusión

    return (accuracy, fail_rate, recall, especificity, precision, npv, f1, confussion_matrix)
end

confusionMatrix (generic function with 2 methods)

In [7]:
function confusionMatrix(outputs::AbstractArray{<:Real,1},
    targets::AbstractArray{Bool,1}; threshold::Real=0.5)

    outputs = outputs .> threshold
    return confusionMatrix(outputs, targets)

end

confusionMatrix (generic function with 2 methods)

## Ejercicio 5

In [3]:
using Random

### Crossvalidation

In [77]:
function crossvalidation(N::Int64, k::Int64)

    folds = 1:k # number of folds
    k_folds = repeat(folds, Int(ceil(N/k))) # number of elements in each fold
    k_folds = k_folds[1:N] # remove the extra elements
    n_folds = shuffle!(k_folds) # shuffle the elements in each fold
    return n_folds

end
    

crossvalidation (generic function with 2 methods)

In [92]:
# Clasificacion binaria

function crossvalidation(targets::AbstractArray{Bool,1}, k::Int64)

   # Asegurarse que cada clase tiene al menos 10 representantes
   min_class_count = min(sum(targets), sum(.!targets))
   if min_class_count < 10     
      return
   end
    
   indices = collect(1:length(targets))
   indices[targets] = crossvalidation(sum(targets), k) # asignar a cada fila un valor de la lista de folds
   indices[.!targets] = crossvalidation(sum(.!targets), k) # Llamar a la funcion anterior con el numero de instancias negativas
   return indices
   
end


crossvalidation (generic function with 3 methods)

In [96]:
# Clasificacion multiclase

function crossvalidation(targets::AbstractArray{Bool,2}, k::Int64)

    # Asegurarse que cada calse tiene al menos 10 representantes
    class_counts = vec(sum(targets, dims=1))  # Número de instancias por clase

    min_class_count = minimum(class_counts)  # Mínimo de instancias en cualquier clase
    if min_class_count < 10
        return
    end

    indices = collect(1:size(targets, 1))
    for i in 1:size(targets, 2)
        indices[targets[:,i]] = crossvalidation(sum(targets[:,i]), k) # asignar a cada fila un valor de la lista de folds
    end

    return indices
end

crossvalidation (generic function with 3 methods)

In [99]:
function crossvalidation(targets::AbstractArray{<:Any,1}, k::Int64) 

    return crossvalidation(oneHotEncoding(targets), k)

end

crossvalidation (generic function with 4 methods)

### Train a K-fold ANN

In [ ]:
function ANNCrossValidation(topology::AbstractArray{<:Int,1},
    dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{<:Any,1}},
    crossValidationIndices::Array{Int64,1};
    numExecutions::Int=50,
    transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)),
    maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01,
    validationRatio::Real=0, maxEpochsVal::Int=20) 


    inputs, targets = dataset # Descomponer dataset
    classes = unique(targets) # Calcular las clases
    one_hot = oneHotEncoding(targets, classes) # OneHot de las clases

    folds = maximum(crossValidationIndices) # Calcular el número de folds


    accuracy = [] # inicializar vector de accuracy
    fail_rate = [] # Inicializar vector de fail_rate
    recall = [] # Inicializar vector de recall
    especificity = [] # Inicializar vector de especifitity
    precision = [] # Inicializar vector de precision
    npv = [] # Inicializar vector de npv
    f1 = [] # Inicializar vector de f1
    confussion_matrix = [0 0; 0 0] # Inicializar confussion matrix

    for fold in 1:folds:

        # Extraer datos de entrenamiento y test según folds
        train_inputs = inputs[:, crossValidationIndices .!= fold] # Extraer inputs de entrenamiento
        train_targets = one_hot[:, crossValidationIndices .!= fold] # Extraer targets de entrenamiento
        test_inputs = inputs[:, crossValidationIndices .== fold] # Extraer inputs de test
        test_targets = one_hot[:, crossValidationIndices .== fold] # Extraer targets de test
        validation_inputs = [] # Inicializar inputs de validación
        validation_targets = [] # Inicializar targets de validación

        if validationRatio > 0 # En caso de que tengamos validación

            v_ratio = validationRatio * (folds/(folds-1)) # Calcular ratio de validación adaptado.

            trainIndices, validationIndices = holdOut(length(train_inputs), validationRatio) # Calcular indices de validación
            validation_inputs = train_inputs[validationIndices, :] # Extraer inputs de validación
            validation_targets = train_targets[validationIndices, :] # Extraer targets de validación
            train_inputs = train_inputs[trainIndices, :] # Extraer inputs de entrenamiento
            train_targets = train_targets[trainIndices, :] # Extraer targets de entrenamiento
        end

        # Crear nuevos vectores para las métricas de cada epoch de cada fold.
        
        acc_folf = []
        fail_rate_fold = []
        recall_fold = []
        especificity_fold = []
        precision_fold = []
        npv_fold = []
        f1_fold = []
        cnf_matrix_fold = Array{Float64}(undef, length(classes), length(classes), numExecutions)

        # Entrenar fold
        for i in 1:numExecutions

            # Entrenar ANN
            ann, trainingLosses, validationLosses, testLosses = trainClassANN(topology, 
            (train_inputs, train_targets);
            validationDataset=(validation_inputs, validation_targets),
            testDataset=(test_inputs, test_targets),
            transferFunctions=transferFunctions, 
            maxEpochs=maxEpochs, 
            minLoss=minLoss, 
            learningRate=learningRate)

            # Calcular métricas
            metrics = confusionMatrix(ann(test_inputs), test_targets, classes)

            # Añadir métricas al registro.
            push!(acc_folf, metrics[1])
            push!(fail_rate_fold, metrics[2])
            push!(recall_fold, metrics[3])
            push!(especificity_fold, metrics[4])
            push!(precision_fold, metrics[5])
            push!(npv_fold, metrics[6])
            push!(f1_fold, metrics[7])
            cnf_matrix_fold[:, :, i] = metrics[8]

        end

        # Calcular métricas de cada fold
        push!(accuracy, mean(acc_folf))
        push!(fail_rate, mean(fail_rate_fold))
        push!(recall, mean(recall_fold))
        push!(especificity, mean(especificity_fold))
        push!(precision, mean(precision_fold))
        push!(npv, mean(npv_fold))
        push!(f1, mean(f1_fold))
        confussion_matrix += dropdims(mean(cnf_matrix_fold, dims=3), dims=3)
    end

    # Devolver media y desviación de las métricas
    return ((mean(acc_folf), std(acc_folf)), (mean(fail_rate_fold), std(fail_rate_fold)), (mean(recall_fold), std(recall_fold)), (mean(especificity_fold), std(especificity_fold)), (mean(precision_fold), std(precision_fold)), (mean(npv_fold), std(npv_fold)), (mean(f1_fold), std(f1_fold)), confussion_matrix)

end